# Connect Claude Code to a Self-Hosted Model on RHOAI

This notebook walks through connecting Claude Code (Anthropic's agentic coding CLI) to a model already deployed via RHOAI's KServe model serving.

## Prerequisites

- A model is already deployed via KServe with the vLLM ServingRuntime (see [deploy-model.md](deploy-model.md) for deployment steps)
- You have the KServe endpoint URL (find it in the RHOAI dashboard under **Model Serving** → your model → **Inference endpoint**)

## What This Notebook Does

1. Grants the workbench access to the KServe endpoint (workbench only)
2. Verifies the model endpoint is reachable
3. Installs Claude Code
4. Configures environment variables
5. Guides you to launch Claude Code from the terminal

## Configuration

Fill in your deployment details below. These values are used throughout the notebook.

In [ ]:
KSERVE_ENDPOINT = "<your-kserve-endpoint>"  # paste the endpoint URL from the RHOAI dashboard (e.g. https://qwen3-coder-my-namespace.apps.my-cluster.openshiftapps.com)
MODEL_NAME = "qwen3-coder"             # must match --served-model-name used during deployment
NAMESPACE = "<namespace>"              # your RHOAI project namespace
WORKBENCH_SA = "<workbench-sa>"        # your workbench service account name (usually matches the workbench name)

---

## Step 1: Grant Workbench Access to the KServe Endpoint

> **This step is only needed if you are running from a RHOAI workbench.** If you are working from your local machine, make sure you are logged into `oc` (OpenShift console → your username → **Copy login command**) and skip to Step 2.

The workbench's service account needs permission to access the KServe endpoint. Without this, requests from the workbench return `Forbidden`.

**This command must be run from a terminal with cluster admin access** (e.g., your local machine), not from this notebook — the workbench service account cannot grant itself roles.

```bash
oc adm policy add-role-to-user view system:serviceaccount:<namespace>:<workbench-sa> -n <namespace>
```

Replace `<namespace>` and `<workbench-sa>` with your values. For example:

```bash
oc adm policy add-role-to-user view system:serviceaccount:claude-code-vllm:claude-code-vllm-1 -n claude-code-vllm
```

---

## Step 2: Verify the Model Endpoint

Run the cell below to confirm you can reach the model. It auto-detects the auth method — service account token on a workbench, `oc whoami -t` locally.

> **You must be logged into `oc` before running this cell.** If you get `Unauthorized`, your token may have expired — re-login via OpenShift console → your username → **Copy login command**.

In [ ]:
import json, urllib.request, ssl, os, subprocess

# Get auth token — service account on workbench, oc whoami -t locally
SA_TOKEN_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/token"

if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        auth_token = f.read().strip()
    print("Using service account token (workbench)")
else:
    result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    auth_token = result.stdout.strip()
    if not auth_token:
        raise RuntimeError("Not on a workbench and 'oc whoami -t' failed — log into oc first")
    print("Using oc token (local)")

# Test the /v1/messages endpoint
url = f"{KSERVE_ENDPOINT}/v1/messages"
data = json.dumps({
    "model": MODEL_NAME,
    "max_tokens": 50,
    "messages": [{"role": "user", "content": "Say hello"}]
}).encode()

req = urllib.request.Request(url, data=data, method="POST", headers={
    "Content-Type": "application/json",
    "Authorization": f"Bearer {auth_token}"
})

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

try:
    with urllib.request.urlopen(req, context=ctx) as resp:
        response = json.loads(resp.read())
        print("Model responded:", response["content"][0]["text"])
except Exception as e:
    print(f"Error: {e}")

---

## Step 3: Install Claude Code

Claude Code is Anthropic's agentic coding CLI. It orchestrates multi-step coding workflows — reading files, editing code, running commands — using an LLM as the backend.

Skip this step if Claude Code is already installed.

In [ ]:
!curl -fsSL https://claude.ai/install.sh | bash

In [ ]:
!claude --version

---

## Step 4: Configure Environment Variables

Claude Code needs to know where to send requests and how to authenticate. The cell below sets all required environment variables.

| Variable | Purpose |
|----------|--------|
| `ANTHROPIC_BASE_URL` | Points Claude Code at the KServe endpoint |
| `ANTHROPIC_AUTH_TOKEN` | Service account token sent as `Bearer` in the `Authorization` header — this is what KServe expects |
| `CLAUDE_CODE_SKIP_AUTH_LOGIN` | Skips Anthropic's login flow |
| `ANTHROPIC_DEFAULT_*_MODEL` | Maps all three model tiers (Opus, Sonnet, Haiku) to the self-hosted model |
| `ANTHROPIC_CUSTOM_MODEL_OPTION` | Bypasses Claude Code's built-in model name validation |
| `CLAUDE_CODE_USE_VERTEX` | Disables Vertex AI routing if configured in the environment |
| `NODE_TLS_REJECT_UNAUTHORIZED` | Allows self-signed TLS certificates (RHOAI routes) |

### Isolating the test config

If you already use Claude Code with Anthropic's hosted API or another provider, set `CLAUDE_CONFIG_DIR` to a throwaway directory so this test doesn't interfere with your existing config:

```bash
mkdir -p /tmp/claude-vllm-test
export CLAUDE_CONFIG_DIR=/tmp/claude-vllm-test
```

Set this **before** the other env vars. All Claude Code data during the test goes to `/tmp/claude-vllm-test` instead of `~/.claude`. When done, close the terminal and optionally delete the directory — your normal Claude Code config remains untouched.

**Note:** The env vars must be set in your **terminal**, not in the notebook kernel — Claude Code runs in the terminal. The cell below prints the `export` commands you can copy-paste.

In [ ]:
import os, subprocess

# Get auth token — service account on workbench, oc whoami -t locally
SA_TOKEN_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/token"

if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        auth_token = f.read().strip()
else:
    result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    auth_token = result.stdout.strip()
    if not auth_token:
        raise RuntimeError("Not on a workbench and 'oc whoami -t' failed — log into oc first")

# Print export commands to copy-paste into the terminal
print("Copy and paste the following into your terminal:")
print("=" * 60)
print(f"""
# Optional — isolate test config from existing Claude Code setup
mkdir -p /tmp/claude-vllm-test
export CLAUDE_CONFIG_DIR=/tmp/claude-vllm-test

export ANTHROPIC_BASE_URL={KSERVE_ENDPOINT}
export ANTHROPIC_AUTH_TOKEN={auth_token}
export CLAUDE_CODE_SKIP_AUTH_LOGIN=1
export NODE_TLS_REJECT_UNAUTHORIZED=0

export ANTHROPIC_DEFAULT_OPUS_MODEL={MODEL_NAME}
export ANTHROPIC_DEFAULT_SONNET_MODEL={MODEL_NAME}
export ANTHROPIC_DEFAULT_HAIKU_MODEL={MODEL_NAME}

export ANTHROPIC_CUSTOM_MODEL_OPTION={MODEL_NAME}
export ANTHROPIC_CUSTOM_MODEL_OPTION_NAME="{MODEL_NAME} on RHOAI"
export ANTHROPIC_CUSTOM_MODEL_OPTION_DESCRIPTION="vLLM on RHOAI KServe"

export CLAUDE_CODE_USE_VERTEX=0
""")
print("=" * 60)

---

## Step 5: Launch Claude Code

Claude Code is an interactive CLI — it requires a live terminal and cannot run inside a notebook cell.

1. Open a **terminal** (on a workbench: File → New → Terminal in JupyterLab)
2. Paste the `export` commands from the cell above
3. Launch Claude Code:

```bash
claude
```

4. Verify the connection with these prompts:

```
What is 2 + 2?
```

```
Write a poem about winter and save it in poem.md
```

If you get responses and the file is created, the setup is complete: **Claude Code → KServe → vLLM → self-hosted model**.

---

## Troubleshooting

| Symptom | Cause | Fix |
|---------|-------|-----|
| `Retrying in Xs · attempt N/10` | Claude Code can't reach the endpoint | Check `ANTHROPIC_BASE_URL`, verify Step 2 works |
| `Unauthorized` | Wrong auth method | Use `ANTHROPIC_AUTH_TOKEN`, not `ANTHROPIC_API_KEY` |
| `Forbidden` | Service account lacks permissions | Run the `oc adm policy` command from Step 1 |
| `Auth conflict` warning | Both `ANTHROPIC_AUTH_TOKEN` and `ANTHROPIC_API_KEY` set | Unset `ANTHROPIC_API_KEY` |
| Login screen appears | `CLAUDE_CODE_SKIP_AUTH_LOGIN` not set or API key format not recognized | Ensure env vars are set in the terminal |
| vLLM OOM crash | Context window too large for available GPU memory | Add `--max-model-len=131072` to serving args |
| `World size larger than available GPUs` | GPU count mismatch | Ensure `--tensor-parallel-size` matches GPU resource limits |